# Predicting Online Purchases
### DSA211 · Seven-model predictive comparison (regularisation revision)

- **Objective:** Choose the strongest observed-purchase classifier by common-fold ROC-AUC, not coefficient inference.
- **Candidates:** Four unchanged operational baselines plus three EDA-informed revisions, including L2-penalised Logit and covariance-shrinkage LDA.
- **Evidence boundary:** Current comparison and voucher illustration use training folds only; previously seen historical holdout is not retested.


## 0. Evaluation protocol

**Purpose.** Compare exactly seven classifier specifications (four unchanged operational baselines plus three EDA-informed revisions), optimising **predictive ranking** rather than coefficient inference.

**Method.** Reproduce the original stratified 80/20 split (seed 42); use the same five training CV folds and mean ROC-AUC for selection, with average precision and fold SD/SE as supporting metrics. All fitted preprocessing is learned on training folds. The revised Logit includes fixed L2 C=1 after fold-only scaling; the revised LDA includes fold-only covariance shrinkage. The three revisions and motivations are in §7.

**Holdout limitation.** Earlier iterations of this project already inspected overlapping test observations. This new candidate comparison is therefore explicitly **training-only**. We do not score a new winner on the reused historical test set, or present the old test score as new-model evidence.

**Policy.** For illustration only, use training out-of-fold predictions at fixed 2/3, a 20% voucher and hypothetical q values. Voucher outcomes do not select the classifier.


In [ ]:
from pathlib import Path
import hashlib, json, warnings, platform, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, scipy, sklearn
from scipy import stats
from scipy.special import logsumexp
from IPython.display import display, Markdown
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.exceptions import ConvergenceWarning
import statsmodels.api as sm
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, roc_curve, confusion_matrix

RUN_START=time.perf_counter()
SEED=42; TEST_FRAC=.20; K=5
TARGET='Revenue'
NUMERIC_ALL=['Administrative','Administrative_Duration','Informational','Informational_Duration',
 'ProductRelated','ProductRelated_Duration','BounceRates','ExitRates','PageValues','SpecialDay']
CAT=['Month','OperatingSystems','Browser','Region','TrafficType','VisitorType']
ACTIVITY=['Administrative','Administrative_Duration','Informational','Informational_Duration','ProductRelated','ProductRelated_Duration']
NUMERIC_SAFE=[x for x in NUMERIC_ALL if x!='PageValues']
ALL=NUMERIC_ALL+CAT+['Weekend']; SAFE=NUMERIC_SAFE+CAT+['Weekend']
EXPECTED=set(ALL+[TARGET])
root_candidates=[Path('data/uci_468.csv'),Path('../data/uci_468.csv')]
data_path=next((p for p in root_candidates if p.is_file()),None)
if data_path is None: raise FileNotFoundError('Run from repository root or a direct child directory; expected data/uci_468.csv')
ROOT=data_path.parent.parent if data_path.parent.name=='data' else data_path.parent.parent
ARTIFACTS=ROOT/'artifacts'; ARTIFACTS.mkdir(exist_ok=True)
WORK=ROOT/'work'; WORK.mkdir(exist_ok=True)
PROGRESS_LOG=WORK/'baseline_progress.jsonl'; PROGRESS_LOG.write_text('')
def mark_progress(candidate,fold,event,**details):
    record={'candidate':candidate,'fold':fold,'event':event,'elapsed_seconds':round(time.perf_counter()-RUN_START,3),**details}
    with PROGRESS_LOG.open('a') as f: f.write(json.dumps(record,default=str)+'\n')
print(f'Python {platform.python_version()} | pandas {pd.__version__} | NumPy {np.__version__} | SciPy {scipy.__version__} | scikit-learn {sklearn.__version__}')
print('Operational selection uses mean five-fold training ROC-AUC; AP and fold variation are supporting evidence. Policy illustrations use only winner training OOF predictions.')

## 1. Data quality and split

**Purpose.** Verify source completeness and establish training/holdout membership.

**Method.** Check missingness, outcome balance and duplicates; preserve all 12,330 sessions. Use training rows only for EDA, candidate selection and threshold comparisons.


In [ ]:
raw=pd.read_csv(data_path); raw.insert(0,'_row_id',np.arange(len(raw),dtype=int))
source_hash=hashlib.sha256(data_path.read_bytes()).hexdigest()
if raw.shape!=(12330,19): raise ValueError(f'Expected 12,330 rows and 18 source columns, found {raw.shape}')
if set(raw.columns)-{'_row_id'} != EXPECTED: raise ValueError(f'Unexpected source columns: {sorted((set(raw.columns)-{"_row_id"}) ^ EXPECTED)}')
if raw[TARGET].isna().any() or not set(raw[TARGET].unique()).issubset({0,1,False,True}): raise ValueError('Revenue must be complete binary 0/1')
raw[TARGET]=raw[TARGET].astype(int)
if raw[NUMERIC_ALL].isna().any().any() or not np.isfinite(raw[NUMERIC_ALL].to_numpy(float)).all(): raise ValueError('Numeric source predictors must be complete and finite')
blank=raw[CAT].astype('string').apply(lambda s:s.str.strip().eq('').sum())
missing=raw.drop(columns='_row_id').isna().sum()
if missing.any() or blank.any(): raise ValueError('Unexpected missing values or blank nominal markers; inspect source before modelling')
if raw['Weekend'].isna().any() or not set(raw['Weekend'].unique()).issubset({0,1,False,True}): raise ValueError('Weekend must be complete binary 0/1')
quality=pd.DataFrame({'dtype':raw.drop(columns='_row_id').dtypes.astype(str),'missing':missing,
                      'blank nominal':blank.reindex(missing.index,fill_value=0),'unique':raw.drop(columns='_row_id').nunique()})
print('Source shape (rows, columns):',(len(raw),len(raw.columns)-1),'SHA-256:',source_hash)
print('Duplicate excess / unique full rows:',int(raw.duplicated(subset=raw.columns.drop('_row_id')).sum()),'/',raw.drop(columns='_row_id').drop_duplicates().shape[0])
print(f'Missing: {int(missing.sum())}; blank nominal: {int(blank.sum())}')
display(raw[TARGET].value_counts().sort_index().rename_axis(TARGET).to_frame('sessions').assign(share=lambda d:d.sessions/len(raw)))
dup=raw.loc[raw.duplicated(subset=raw.columns.drop('_row_id'),keep='first')]
print(f'Excess duplicates: {len(dup)}; duplicate outcomes:',dup[TARGET].value_counts().sort_index().to_dict())

train_ids,test_ids=train_test_split(raw['_row_id'].to_numpy(),test_size=TEST_FRAC,
    stratify=raw[TARGET].to_numpy(),random_state=SEED)
train_ids=np.sort(train_ids); test_ids=np.sort(test_ids)
pd.DataFrame({'row_id':train_ids,'partition':'train'}).to_csv(ARTIFACTS/'train_row_ids.csv',index=False)
pd.DataFrame({'row_id':test_ids,'partition':'test'}).to_csv(ARTIFACTS/'test_row_ids.csv',index=False)
train=raw.set_index('_row_id').loc[train_ids].copy()
y_train=train.pop(TARGET).astype(int)
X_train=train[ALL].copy()
X_train[CAT]=X_train[CAT].astype('string')
X_train['Weekend']=X_train['Weekend'].astype(int)
del raw,dup
cv=StratifiedKFold(n_splits=K,shuffle=True,random_state=SEED)
FOLD_INDICES=list(cv.split(np.zeros(len(y_train)),y_train))
fold_map=[]
for fold,(fit_idx,val_idx) in enumerate(FOLD_INDICES,1):
    fold_map += [{'row_id':int(X_train.index[i]),'fold':fold,'role':'fit'} for i in fit_idx]
    fold_map += [{'row_id':int(X_train.index[i]),'fold':fold,'role':'validation'} for i in val_idx]
pd.DataFrame(fold_map).to_csv(ARTIFACTS/'cv_fold_row_ids.csv',index=False)
print(f'Training={len(X_train):,}; holdout IDs recorded={len(test_ids):,}; purchases={y_train.sum():,} ({y_train.mean():.2%}).')


## 2. Numerical EDA

**Purpose.** Identify skew and excess zeros.

**Method.** Display median, zero share and skewness, with representative histograms.

**Implication.** Test log1p for long tails and fold-fitted presence indicators for zero-heavy activity.


In [ ]:
Xtr=X_train; ytr=y_train
num_summary=pd.DataFrame({'median':Xtr[NUMERIC_ALL].median(),'zero share':Xtr[NUMERIC_ALL].eq(0).mean(),
                          'skew':Xtr[NUMERIC_ALL].skew(),'buyer median':Xtr.loc[ytr.eq(1),NUMERIC_ALL].median(),
                          'nonbuyer median':Xtr.loc[ytr.eq(0),NUMERIC_ALL].median()})
display(num_summary[['median','zero share','skew']].round(3))
fig,axes=plt.subplots(1,2,figsize=(8,3.5))
for ax,col in zip(axes.flat,['Informational_Duration','ProductRelated_Duration']):
    ax.hist(Xtr[col],bins=35,log=True,color='#4779a8'); ax.set(title=col,xlabel='Recorded value',ylabel='Training sessions (log count)')
plt.tight_layout(); plt.show()
long_tail=num_summary.loc[ACTIVITY,'skew']


## 3. Categorical EDA

**Purpose.** Compare observed buying rates across nominal levels.

**Method.** Summarise sparsity and show Month/VisitorType rates on training rows.

**Implication.** Encode nominal levels in each CV fit; avoid over-interpreting small groups.


In [ ]:
cat_summary=[]
for col in CAT:
    tab=Xtr.assign(Revenue=ytr).groupby(col,observed=True).agg(sessions=('Revenue','size'),buyers=('Revenue','sum'),buyer_rate=('Revenue','mean'))
    cat_summary.append({'feature':col,'levels':len(tab),'small levels <50':int((tab.sessions<50).sum()),
      'small-level sessions':int(tab.loc[tab.sessions<50,'sessions'].sum()),
      'buyer-rate range':f"{tab.buyer_rate.min():.1%}–{tab.buyer_rate.max():.1%}"})
display(pd.DataFrame(cat_summary).set_index('feature')[['levels','small levels <50','buyer-rate range']])
for col in ['Month','VisitorType']:
    tab=Xtr.assign(Revenue=ytr).groupby(col,observed=True).agg(sessions=('Revenue','size'),buyers=('Revenue','sum'),buyer_rate=('Revenue','mean')).sort_values('sessions',ascending=False)
    print(f'Training group counts and observed buyer rates: {col}')
    display(tab[['sessions','buyer_rate']].round(3))
rare=[]
for col in ['TrafficType','Browser','Region']:
    tab=Xtr.assign(Revenue=ytr).groupby(col,observed=True).agg(sessions=('Revenue','size'),buyers=('Revenue','sum'),buyer_rate=('Revenue','mean'))
    small=tab[tab.sessions<50]
    rare.append({'feature':col,'levels below n=50':len(small),'total levels':len(tab),
                 'rare-cell rate range':f"{small.buyer_rate.min():.1%}–{small.buyer_rate.max():.1%}" if len(small) else 'none'})
vtm=Xtr.assign(Revenue=ytr).groupby(['VisitorType','Month'],observed=True).agg(sessions=('Revenue','size'),buyers=('Revenue','sum'),buyer_rate=('Revenue','mean')).reset_index()
special=Xtr.assign(Revenue=ytr).groupby('SpecialDay',observed=True).agg(sessions=('Revenue','size'),buyers=('Revenue','sum'),buyer_rate=('Revenue','mean'))


## 4. Predictor relationships and assumptions

**Purpose.** Assess dependence and departure from Gaussian shapes.

**Method.** Check selected within-class correlations and Q–Q plots.

**Implication.** Diagnostics motivate candidates; CV, not visual assumptions alone, evaluates performance.


In [ ]:
NUM_SAFE=[c for c in NUMERIC_ALL if c!='PageValues']+['Weekend']
for cls in [0,1]:
    corr=Xtr.loc[ytr.eq(cls),NUM_SAFE].corr().abs()
    pairs=corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool)).stack().sort_values(ascending=False).head(5)
print('BounceRates/ExitRates correlations are {:.3f} for nonbuyers and {:.3f} for buyers; ProductRelated/ProductRelated_Duration are {:.3f} and {:.3f}. These selected differences do not establish unequal full covariance matrices.'.format(
 Xtr.loc[ytr.eq(0),['BounceRates','ExitRates']].corr().iloc[0,1],Xtr.loc[ytr.eq(1),['BounceRates','ExitRates']].corr().iloc[0,1],
 Xtr.loc[ytr.eq(0),['ProductRelated','ProductRelated_Duration']].corr().iloc[0,1],Xtr.loc[ytr.eq(1),['ProductRelated','ProductRelated_Duration']].corr().iloc[0,1]))
fig,axes=plt.subplots(1,2,figsize=(8,3.5))
for ax,col in zip(axes,['ProductRelated_Duration','Informational_Duration']):
    for cls,label,color in [(0,'No purchase','#4779a8'),(1,'Purchase','#d97941')]:
        vals=Xtr.loc[ytr.eq(cls),col].to_numpy()
        (osm,osr),_=stats.probplot(vals,dist='norm',fit=True)
        ax.plot(osm,osr,'.',ms=2,alpha=.35,label=label,color=color)
    ax.set(title=f'{col}: normal Q–Q by class',xlabel='Normal theoretical quantile',ylabel='Observed quantile'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

def marginal_elogit(col, bins=8):
    z=pd.DataFrame({'x':Xtr[col],'buyer':ytr}); z['bin']='zero'; positive=z.x.gt(0)
    if positive.any(): z.loc[positive,'bin']=pd.qcut(z.loc[positive,'x'],bins,duplicates='drop').astype(str)
    out=z.groupby('bin',observed=True).agg(n=('buyer','size'),mean_x=('x','mean'),buyers=('buyer','sum'),rate=('buyer','mean')).reset_index()
    out['empirical_logit']=np.log((out.buyers+.5)/(out.n-out.buyers+.5))
    return out.sort_values('mean_x')


In [ ]:
class BinaryMagnitude(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None): return self
    def transform(self,X): return (np.asarray(X,dtype=float)>0).astype(float)

def make_preprocessor(columns, family):
    nums=[x for x in NUMERIC_ALL if x in columns]
    cats=[x for x in CAT if x in columns]
    if family=='BernoulliNB': num_transform=BinaryMagnitude()
    else: num_transform='passthrough'
    drop='first' if family in ['Logit','LDA'] else None
    return ColumnTransformer([('numeric',num_transform,nums),
      ('categorical',OneHotEncoder(handle_unknown='ignore',drop=drop,sparse_output=False),cats),
      ('weekend','passthrough',['Weekend'])],remainder='drop',sparse_threshold=0.0,verbose_feature_names_out=False)

def make_estimator(family):
    if family=='Logit': return LogisticRegression(C=np.inf,solver='newton-cg',tol=1e-8,max_iter=2000)
    if family=='GaussianNB': return GaussianNB()
    if family=='BernoulliNB': return BernoulliNB(binarize=None)
    if family=='LDA': return LinearDiscriminantAnalysis(solver='svd')
    raise ValueError(family)

def make_pipeline(columns,family,estimator=None):
    return Pipeline([('pre',make_preprocessor(columns,family)),('clf',clone(estimator) if estimator is not None else make_estimator(family))])

CONFIGS={
 'Logit':{'family':'Logit','columns':ALL},
 'GaussianNB':{'family':'GaussianNB','columns':ALL},
 'BernoulliNB':{'family':'BernoulliNB','columns':ALL},
 'LDA':{'family':'LDA','columns':ALL}}

def pure_category_levels(X,y,cats):
    result=[]
    for col in cats:
        tab=pd.DataFrame({'level':X[col].astype(str).to_numpy(),'y':np.asarray(y)}).groupby('level').y.agg(['size','sum'])
        pure=tab[(tab['sum']==0)|(tab['sum']==tab['size'])]
        for level,r in pure.iterrows(): result.append(f'{col}={level}:n{int(r["size"])}:y{int(r["sum"])}')
    return result

def design_diagnostics(pre,Xfit):
    Z=np.asarray(pre.transform(Xfit),dtype=float)
    rank=int(np.linalg.matrix_rank(Z)); p=Z.shape[1]
    std=Z.std(axis=0); keep=std>1e-12
    Zn=(Z[:,keep]-Z[:,keep].mean(axis=0))/std[keep] if keep.any() else np.empty((len(Z),0))
    cond=float(np.linalg.cond(np.column_stack([np.ones(len(Z)),Zn]))) if Zn.shape[1] else np.nan
    return Z,rank,p,cond

def positive_probability(est,Z):
    classes=np.asarray(est.classes_); idx=np.flatnonzero(classes==1)
    if len(idx)!=1: raise ValueError(f'Expected one positive class 1, found {classes}')
    return np.asarray(est.predict_proba(Z)[:,idx[0]],dtype=float)

def _finite_numeric_tree(value):
    if isinstance(value,(list,tuple)):
        return all(_finite_numeric_tree(x) for x in value)
    try:
        arr=np.asarray(value,dtype=float)
        return bool(np.isfinite(arr).all())
    except (TypeError,ValueError):
        return False

def finite_fitted_parameters(est):
    names=['coef_','intercept_','theta_','var_','class_prior_','class_log_prior_','feature_log_prob_',
           'scalings_','means_','xbar_','priors_','cat_log_probs_','class_count_']
    checked={name:_finite_numeric_tree(getattr(est,name)) for name in names if hasattr(est,name)}
    if hasattr(est,'gaussian_'):
        for name in ['theta_','var_','class_prior_','class_log_prior_']:
            if hasattr(est.gaussian_,name): checked['gaussian_'+name]=_finite_numeric_tree(getattr(est.gaussian_,name))
    return bool(checked) and all(checked.values()),checked

def logit_nll(est,Z,y):
    decision=np.asarray(est.decision_function(Z),dtype=float)
    return float(np.sum(np.logaddexp(0.0,decision)-np.asarray(y,dtype=float)*decision))

def unknown_category_counts(Xfit,Xval,cats):
    out={}
    for col in cats:
        known=set(Xfit[col].astype(str)); unseen=~Xval[col].astype(str).isin(known)
        out[col]={'count':int(unseen.sum()),'levels':sorted(Xval.loc[unseen,col].astype(str).unique().tolist())}
    return out

class RevisionPreprocessor(TransformerMixin,BaseEstimator):
    """Fold-fitted log1p activity block, optional zero-heavy presence, and k-1 OHE."""
    def __init__(self,add_presence=False,standardize=False):
        self.add_presence=add_presence
        self.standardize=standardize
    def fit(self,X,y=None):
        self.zero_share_={c:float(X[c].eq(0).mean()) for c in ACTIVITY}
        self.presence_columns_=[c for c in ACTIVITY if self.add_presence and self.zero_share_[c]>=.50]
        self.numeric_columns_=[*ACTIVITY,*[c for c in NUMERIC_SAFE if c not in ACTIVITY],'Weekend']
        self.feature_columns_=[*self.numeric_columns_,*[f'{c}__present' for c in self.presence_columns_],*CAT]
        features=self._represent(X)
        self.encoded_numeric_columns_=[c for c in features.columns if c not in CAT]
        self.encoder_=ColumnTransformer([('numeric','passthrough',self.encoded_numeric_columns_),
            ('categorical',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),CAT)],remainder='drop')
        self.encoder_.fit(features)
        if self.standardize:
            self.scaler_=StandardScaler().fit(np.asarray(self.encoder_.transform(features),dtype=float))
        self.n_features_in_=X.shape[1]
        self.feature_names_in_=np.asarray(list(X.columns),dtype=object)
        return self
    def _represent(self,X):
        out=pd.DataFrame(index=X.index)
        for c in ACTIVITY: out[c]=np.log1p(X[c].astype(float))
        for c in NUMERIC_SAFE:
            if c not in ACTIVITY: out[c]=X[c].astype(float)
        out['Weekend']=X['Weekend'].astype(float)
        for c in self.presence_columns_: out[f'{c}__present']=X[c].gt(0).astype(float)
        for c in CAT: out[c]=X[c].astype('string')
        return out
    def transform(self,X):
        Z=np.asarray(self.encoder_.transform(self._represent(X)),dtype=float)
        return self.scaler_.transform(Z) if self.standardize else Z
    def get_feature_names_out(self,input_features=None):
        return self.encoder_.get_feature_names_out()

class IdentityFrameTransformer(TransformerMixin,BaseEstimator):
    """Preserve source columns for the mixed-likelihood NB estimator."""
    def fit(self,X,y=None):
        self.feature_names_in_=np.asarray(list(X.columns),dtype=object); self.n_features_in_=len(self.feature_names_in_); return self
    def transform(self,X): return X.loc[:,list(self.feature_names_in_)].copy()

class MixedGaussianCategoricalNB(ClassifierMixin,BaseEstimator):
    """Eight Gaussian numeric inputs plus eight smoothed categorical likelihoods."""
    def __init__(self,alpha=1.0):
        self.alpha=alpha
    def _numeric(self,X):
        out=pd.DataFrame(index=X.index)
        for c in ACTIVITY: out[c]=np.log1p(X[c].astype(float))
        out['BounceRates']=X['BounceRates'].astype(float)
        out['ExitRates']=X['ExitRates'].astype(float)
        return out.to_numpy(dtype=float)
    def _categorical(self,X):
        out=pd.DataFrame(index=X.index)
        for c in CAT: out[c]=X[c].astype(str)
        out['Weekend']=pd.to_numeric(X['Weekend'],errors='raise').astype(int).astype(str)
        out['SpecialDay__present']=X['SpecialDay'].gt(0).astype(int).astype(str)
        return out
    def fit(self,X,y):
        y=np.asarray(y,dtype=int)
        self.gaussian_=GaussianNB().fit(self._numeric(X),y)
        self.classes_=self.gaussian_.classes_.copy(); self.class_count_=np.asarray(self.gaussian_.class_count_,dtype=float)
        self.class_prior_=self.class_count_/self.class_count_.sum(); self.class_log_prior_=np.log(self.class_prior_)
        self.category_maps_={}; self.category_counts_=[]; self.category_support_={}; self.alpha_denominators_=[]; self.cat_log_probs_=[]; cat=self._categorical(X)
        class_index={v:i for i,v in enumerate(self.classes_)}
        for col in cat.columns:
            levels=sorted(cat[col].astype(str).unique().tolist())
            mapping={value:i+1 for i,value in enumerate(levels)}
            self.category_maps_[col]=mapping
            support=len(levels)+1
            self.category_support_[col]={'unknown_code':0,'known_levels':levels,'support_size':support} # code 0 is reserved unknown support, even when not observed during fit
            counts=np.zeros((len(self.classes_),support),dtype=float)
            codes=cat[col].astype(str).map(mapping).fillna(0).to_numpy(dtype=int)
            for yi,code in zip(y,codes): counts[class_index[yi],code]+=1.0
            denom=self.class_count_+float(self.alpha)*support
            self.category_counts_.append(counts)
            self.alpha_denominators_.append(denom.copy())
            self.cat_log_probs_.append(np.log((counts+float(self.alpha))/denom[:,None]))
        self.n_features_in_=len(X.columns); self.feature_names_in_=np.asarray(list(X.columns),dtype=object)
        return self
    def _scores(self,X):
        scores=np.asarray(self.gaussian_.predict_log_proba(self._numeric(X)),dtype=float)
        cat=self._categorical(X)
        for col,log_prob in zip(cat.columns,self.cat_log_probs_):
            mapping=self.category_maps_[col]
            codes=cat[col].astype(str).map(mapping).fillna(0).to_numpy(dtype=int)
            scores+=log_prob[:,codes].T
        return scores-logsumexp(scores,axis=1,keepdims=True)
    def predict_log_proba(self,X): return self._scores(X)
    def predict_proba(self,X): return np.exp(self._scores(X))
    def predict(self,X): return self.classes_[np.argmax(self._scores(X),axis=1)]
    def component_audit(self,Xval):
        catval=self._categorical(Xval); numval=self._numeric(Xval)
        positions=[0] if len(Xval) else []
        unknown_positions=[]
        for j,col in enumerate(catval.columns):
            mapping=self.category_maps_[col]
            mask=~catval[col].astype(str).isin(mapping)
            unknown_positions.extend(np.flatnonzero(mask.to_numpy()).tolist())
        if unknown_positions:
            first_unknown=min(unknown_positions)
            if first_unknown not in positions: positions.append(first_unknown)
        gaussian_rows=[]
        for i,x in enumerate(numval):
            by_class=[]
            for k,cls in enumerate(self.classes_):
                log_density=float(np.sum(-.5*(np.log(2*np.pi*self.gaussian_.var_[k])+((x-self.gaussian_.theta_[k])**2/self.gaussian_.var_[k]))))
                by_class.append({'class':int(cls),'log_density':log_density,'log_prior':float(self.class_log_prior_[k])})
            gaussian_rows.append(by_class)
        samples=[]
        for pos in positions:
            codes={};cat_ll=np.zeros(len(self.classes_),dtype=float)
            raw_cat={}
            for j,col in enumerate(catval.columns):
                val=str(catval.iloc[pos][col]); mapping=self.category_maps_[col]
                code=int(mapping.get(val,0)); codes[col]=code;raw_cat[col]=val
                cat_ll+=self.cat_log_probs_[j][:,code]
            joint=np.asarray([gaussian_rows[pos][k]['log_density']+gaussian_rows[pos][k]['log_prior']+cat_ll[k]
                              for k in range(len(self.classes_))],dtype=float)
            posterior=np.exp(joint-logsumexp(joint))
            raw_numeric={c:float(Xval.iloc[pos][c]) for c in [*ACTIVITY,'BounceRates','ExitRates']}
            transformed_numeric={c:float(v) for c,v in zip([*ACTIVITY,'BounceRates','ExitRates'],numval[pos])}
            samples.append({'row_id':int(Xval.index[pos]),'raw_numeric':raw_numeric,'transformed_numeric':transformed_numeric,
                'categorical_values':raw_cat,'categorical_codes':codes,
                'gaussian_class_components':gaussian_rows[pos],
                'sum_categorical_log_likelihood_by_class':{str(int(c)):float(cat_ll[k]) for k,c in enumerate(self.classes_)},
                'joint_log_score_by_class':{str(int(c)):float(joint[k]) for k,c in enumerate(self.classes_)},
                'posterior_probability_by_class':{str(int(c)):float(posterior[k]) for k,c in enumerate(self.classes_)}})
        return {'classes':self.classes_.astype(int).tolist(),'class_count':self.class_count_.astype(float).tolist(),
          'class_prior':self.class_prior_.astype(float).tolist(),'gaussian_features':[*ACTIVITY,'BounceRates','ExitRates'],
          'gaussian_theta':self.gaussian_.theta_.astype(float).tolist(),'gaussian_var':self.gaussian_.var_.astype(float).tolist(),
          'categorical_alpha':float(self.alpha),'categorical_fields':[
            {'field':col,'mapping':self.category_maps_[col],'support':self.category_support_[col],
             'counts_by_class':self.category_counts_[j].astype(float).tolist(),
             'alpha_denominator_by_class':self.alpha_denominators_[j].astype(float).tolist()}
             for j,col in enumerate(self.category_maps_)],
          'validation_component_examples':samples}

    def unseen_level_counts(self,X):
        cat=self._categorical(X); out={}
        for col,mapping in self.category_maps_.items():
            mask=~cat[col].astype(str).isin(mapping)
            out[col]={'count':int(mask.sum()),'levels':sorted(cat.loc[mask,col].astype(str).unique().tolist())}
        return out

def make_revision_pipeline(kind):
    if kind in ['Logit','LDA']:
        pre=RevisionPreprocessor(add_presence=(kind=='Logit'),standardize=(kind=='Logit'))
        clf=(LogisticRegression(C=1.0,solver='lbfgs',tol=1e-8,max_iter=3000)
             if kind=='Logit' else LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'))
        return Pipeline([('pre',pre),('clf',clf)])
    if kind=='MixedNB': return Pipeline([('pre',IdentityFrameTransformer()),('clf',MixedGaussianCategoricalNB(alpha=1.0))])
    raise ValueError(kind)

def compact_numerical_label(result):
    value=result.get('numerical_status','UNKNOWN')
    if value.startswith('PURE_LEVEL_BOUNDARY_MLE_IRLS_GATE_PASSED'):
        return 'Boundary MLE; IRLS reference stable'
    if value=='IRLS_NUMERICAL_REFERENCE_GATE_FAILED':
        return 'IRLS reference gate failed'
    if value=='FINITE_WITH_WARNINGS_REVIEW_REQUIRED':
        return 'Finite fit; warnings logged'
    if value=='FINITE_PARAMETERS_AND_PROBABILITIES_REVIEW_REQUIRED':
        return 'Finite fit; assumptions apply'
    return value

def run_spec(label,family,columns,pipeline_factory=None):
    folds=[]; oof=np.full(len(ytr),np.nan); warning_rows=[]
    cats=[x for x in CAT if x in columns]
    for fold,(fit_idx,val_idx) in enumerate(FOLD_INDICES,1):
        fold_start=time.perf_counter(); mark_progress(label,fold,'fold_start',n_fit=len(fit_idx),n_validation=len(val_idx))
        Xfit=Xtr.iloc[fit_idx][columns]; yfit=ytr.iloc[fit_idx]
        Xval=Xtr.iloc[val_idx][columns]; yval=ytr.iloc[val_idx]
        pipe=pipeline_factory() if pipeline_factory is not None else make_pipeline(columns,family)
        caught=[]; rank=p=None; cond=np.nan
        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                # Preprocessing, estimator fitting and validation transform/prediction are all fold-contained.
                pipe.fit(Xfit,yfit)
                fitted_est=pipe.named_steps['clf']
                mark_progress(label,fold,'fit_complete',fit_seconds=round(time.perf_counter()-fold_start,3),
                    n_iter=int(np.max(fitted_est.n_iter_)) if hasattr(fitted_est,'n_iter_') else None,
                    max_abs_coefficient=float(np.max(np.abs(fitted_est.coef_))) if hasattr(fitted_est,'coef_') else None)
                pre=pipe.named_steps['pre']; est=pipe.named_steps['clf']
                diagnostics_start=time.perf_counter(); mark_progress(label,fold,'design_diagnostics_start')
                if family=='MixedNB':
                    Zfit=pre.transform(Xfit); Zval=pre.transform(Xval)
                    rank=np.nan; p=np.nan; cond=np.nan
                else:
                    Zfit,rank,p,cond=design_diagnostics(pre,Xfit)
                    Zval=np.asarray(pre.transform(Xval),dtype=float)
                mark_progress(label,fold,'design_diagnostics_complete',diagnostics_seconds=round(time.perf_counter()-diagnostics_start,3),rank=rank,columns=p,condition=cond)
                prob=positive_probability(est,Zval)
            finite_params,parameter_checks=finite_fitted_parameters(est)
            finite_prob=bool(np.isfinite(prob).all())
            prob_in_range=bool(finite_prob and np.all((prob>=0)&(prob<=1)))
            if not finite_params: raise ValueError(f'non-finite fitted parameters: {parameter_checks}')
            if not prob_in_range: raise ValueError('positive-class probabilities are non-finite or outside [0,1]')
            oof[val_idx]=prob
            pure=pure_category_levels(Xfit,yfit,cats)
            unknown=unknown_category_counts(Xfit,Xval,cats)
            if family=='MixedNB': unknown.update({'MixedNB '+k:v for k,v in est.unseen_level_counts(Xval).items()})
            row={'candidate':label,'family':family,'fold':fold,'fit_status':'OK','n_fit':len(fit_idx),'n_validation':len(val_idx),
                 'roc_auc':roc_auc_score(yval,prob),'average_precision':average_precision_score(yval,prob),
                 'design_rank':rank,'design_columns':p,'normalized_condition_number':cond,
                 'finite_model_parameters':finite_params,'parameter_checks':json.dumps(parameter_checks),
                 'finite_probabilities':finite_prob,'probabilities_in_0_1':prob_in_range,
                 'probability_min':float(prob.min()),'probability_max':float(prob.max()),
                 'max_abs_coefficient':float(np.max(np.abs(est.coef_))) if hasattr(est,'coef_') else np.nan,
                 'n_iter':int(np.max(est.n_iter_)) if hasattr(est,'n_iter_') else np.nan,
                 'training_nll':logit_nll(est,Zfit,yfit) if family=='Logit' else np.nan,
                 'pure_nominal_count':len(pure),'pure_nominal_levels':'; '.join(pure),
                 'unseen_validation_levels':json.dumps(unknown),
                 'fold_zero_share':json.dumps(getattr(pre,'zero_share_',{})),
                 'fold_presence_columns':json.dumps(getattr(pre,'presence_columns_',[])),
                 'mixed_nb_component_audit':est.component_audit(Xval) if family=='MixedNB' else None}
            if family=='Logit' and np.isinf(est.C):
                # Only unpenalised Logit has the same statistical objective as the IRLS reference.
                # Admission is based only on exact training NLL and label-free validation probabilities.
                reference_warn=[]; reference_error=''; ref=None
                Afit=np.column_stack([np.ones(len(Zfit)),Zfit]); Aval=np.column_stack([np.ones(len(Zval)),Zval])
                start=np.r_[float(est.intercept_[0]),np.asarray(est.coef_[0],dtype=float)]
                with warnings.catch_warnings(record=True) as reference_warn:
                    warnings.simplefilter('always')
                    try:
                        ref=sm.GLM(yfit.to_numpy(),Afit,family=sm.families.Binomial()).fit(
                            start_params=start,maxiter=200,tol=1e-10,method='IRLS',wls_method='pinv',disp=0)
                        ref_params=np.asarray(ref.params,dtype=float)
                        ref_eta_fit=Afit@ref_params; ref_eta_val=Aval@ref_params
                        ref_prob=np.asarray(ref.predict(Aval),dtype=float)
                        ref_nll=float(np.mean(np.logaddexp(0.0,ref_eta_fit)-yfit.to_numpy()*ref_eta_fit))
                        primary_mean_nll=float(row['training_nll']/len(yfit))
                        ref_finite=bool(np.isfinite(ref_params).all() and np.isfinite(ref_eta_fit).all() and np.isfinite(ref_eta_val).all()
                            and np.isfinite(ref_prob).all() and np.all((ref_prob>=0)&(ref_prob<=1)))
                        nll_delta=float(abs(ref_nll-primary_mean_nll))
                        p_delta=float(np.max(np.abs(ref_prob-prob)))
                        ref_converged=bool(ref.converged)
                    except Exception as ref_exc:
                        reference_error=f'{type(ref_exc).__name__}: {str(ref_exc)[:250]}'
                        ref_finite=False; nll_delta=np.nan; p_delta=np.nan; ref_converged=False
                for w in reference_warn:
                    warning_rows.append({'candidate':label,'fold':fold,'warning_type':'IRLS '+w.category.__name__,'message':str(w.message)[:300]})
                reference_pass=bool(ref_finite and ref_converged and nll_delta<=1e-6 and p_delta<=1e-3)
                row.update({'reference_solver':'statsmodels GLM Binomial IRLS','reference_wls_method':'pinv',
                    'reference_tol':1e-10,'reference_maxiter':200,
                    'reference_iterations':int(ref.fit_history.get('iteration',-1)) if ref is not None else np.nan,
                    'reference_converged':ref_converged,'reference_finite_parameters_and_probabilities':ref_finite,
                    'reference_mean_training_nll':ref_nll if ref_finite else np.nan,
                    'reference_abs_mean_training_nll_delta':nll_delta,'reference_max_abs_validation_probability_delta':p_delta,
                    'reference_passes_fixed_criteria':reference_pass,'reference_error':reference_error,
                    'reference_warnings':json.dumps([{'type':w.category.__name__,'message':str(w.message)} for w in reference_warn])})
            if family=='Logit' and np.isfinite(est.C):
                # The unpenalised IRLS reference is inapplicable to an L2-penalised objective.
                row['reference_passes_fixed_criteria']=None
                row['reference_solver']='not applicable: L2 penalised objective'
                if any(issubclass(w.category,ConvergenceWarning) for w in caught):
                    raise RuntimeError('Regularised Logit did not converge on this fold')
                row['numerical_status']='L2_REGULARISED_FINITE_FIT'
            elif family=='Logit':
                if not row.get('reference_passes_fixed_criteria',False):
                    row['numerical_status']='IRLS_NUMERICAL_REFERENCE_GATE_FAILED'
                elif row['pure_nominal_count']>0:
                    row['numerical_status']='PURE_LEVEL_BOUNDARY_MLE_IRLS_GATE_PASSED_NO_FINITE_MLE_CERTIFICATION'
                else:
                    row['numerical_status']='IRLS_NUMERICAL_REFERENCE_GATE_PASSED'
            else:
                row['numerical_status']='FINITE_PARAMETERS_AND_PROBABILITIES_REVIEW_REQUIRED'
            for w in caught: warning_rows.append({'candidate':label,'fold':fold,'warning_type':w.category.__name__,'message':str(w.message)[:300]})
            folds.append(row)
            mark_progress(label,fold,'fold_complete',fold_seconds=round(time.perf_counter()-fold_start,3),
                roc_auc=row['roc_auc'],numerical_status=row['numerical_status'])
        except Exception as exc:
            for w in caught: warning_rows.append({'candidate':label,'fold':fold,'warning_type':w.category.__name__,'message':str(w.message)[:300]})
            mark_progress(label,fold,'fold_failed',fold_seconds=round(time.perf_counter()-fold_start,3),failure=f'{type(exc).__name__}: {str(exc)[:250]}')
            folds.append({'candidate':label,'family':family,'fold':fold,'n_fit':len(fit_idx),'n_validation':len(val_idx),
                          'design_rank':rank,'design_columns':p,'normalized_condition_number':cond,
                          'fit_status':'FAILED','failure':f'{type(exc).__name__}: {str(exc)[:250]}'})
    complete=bool(np.isfinite(oof).all()) and len(folds)==K and all(r.get('fit_status')=='OK' for r in folds)
    statuses=[r.get('numerical_status','FAILED') for r in folds]
    if not complete: numerical_status='SCORING_FAILED'
    elif family=='Logit' and 'IRLS_NUMERICAL_REFERENCE_GATE_FAILED' in statuses: numerical_status='IRLS_NUMERICAL_REFERENCE_GATE_FAILED'
    elif family=='Logit' and 'L2_REGULARISED_FINITE_FIT' in statuses: numerical_status='L2_REGULARISED_FINITE_FIT'
    elif family=='Logit' and any('PURE_LEVEL_BOUNDARY_MLE' in x for x in statuses): numerical_status='PURE_LEVEL_BOUNDARY_MLE_IRLS_GATE_PASSED_NO_FINITE_MLE_CERTIFICATION'
    elif family=='Logit': numerical_status='IRLS_NUMERICAL_REFERENCE_GATE_PASSED'
    elif warning_rows: numerical_status='FINITE_WITH_WARNINGS_REVIEW_REQUIRED'
    else: numerical_status='FINITE_PARAMETERS_AND_PROBABILITIES_REVIEW_REQUIRED'
    result={'candidate':label,'family':family,'stage':'all17' if len(columns)==len(ALL) else 'operational16',
            'status':'SCORED' if complete else 'FAILED','numerical_status':numerical_status,'folds':folds,'oof':oof if complete else None,
            'warnings':warning_rows,'failure':'' if complete else 'one or more folds failed or OOF scores are incomplete'}
    if complete:
        auc=np.array([r['roc_auc'] for r in folds]); ap=np.array([r['average_precision'] for r in folds])
        result.update({'mean_fold_auc':auc.mean(),'fold_auc_sd':auc.std(ddof=1),'fold_auc_se':auc.std(ddof=1)/np.sqrt(K),
                       'mean_fold_ap':ap.mean(),'fold_ap_sd':ap.std(ddof=1),'fold_ap_se':ap.std(ddof=1)/np.sqrt(K),
                       'pooled_oof_auc':roc_auc_score(ytr,oof),'pooled_oof_ap':average_precision_score(ytr,oof)})
    return result

RESULTS={}; FOLD_ROWS=[]; WARNING_ROWS=[]


## 5. Four all-17 first-pass models

**Purpose.** Start with every source predictor.

**Method.** Compare Logit, GaussianNB, BernoulliNB and LDA using common training folds.

**Implication.** This is a diagnostic pass, not the operational model selection.


In [ ]:
# A5 — diagnostic first pass, all 17 predictors
for family in ['Logit','GaussianNB','BernoulliNB','LDA']:
    label=f'All17 | {family}'
    print('Fitting',label,flush=True)
    RESULTS[label]=run_spec(label,family,ALL)
    FOLD_ROWS.extend(RESULTS[label]['folds']); WARNING_ROWS.extend(RESULTS[label]['warnings'])
ALL17=pd.DataFrame([{'candidate':r['candidate'],'scoring status':r['status'],'numerical status':compact_numerical_label(r),'mean ROC-AUC':r.get('mean_fold_auc',np.nan),
 'AUC SD':r.get('fold_auc_sd',np.nan),'AUC SE':r.get('fold_auc_se',np.nan),'mean AP':r.get('mean_fold_ap',np.nan),
 'pooled OOF AUC':r.get('pooled_oof_auc',np.nan),'warnings':len(r['warnings']),'failure':r['failure']} for r in RESULTS.values() if r['stage']=='all17']).set_index('candidate')
display(ALL17[['mean ROC-AUC','AUC SD','mean AP']].round(4))
fold_diag=pd.DataFrame([x for r in RESULTS.values() if r['stage']=='all17' for x in r['folds']])
all17_audit=fold_diag.groupby('candidate').agg(rank_min=('design_rank','min'),rank_max=('design_rank','max'),
    encoded_columns_min=('design_columns','min'),encoded_columns_max=('design_columns','max'),
    all_parameters_finite=('finite_model_parameters','all'),all_probabilities_in_range=('probabilities_in_0_1','all'))
logit_diag=fold_diag[fold_diag.family.eq('Logit')].copy()


**Interpretation.** `PageValues` may reflect information unavailable when a voucher must be offered. Exclude it from operational models; a large AUC difference alone does not establish leakage.


In [ ]:
# PageValues descriptive association on training rows only
pv=Xtr.assign(Revenue=ytr).groupby(Xtr.PageValues.gt(0).map({False:'PageValues = 0',True:'PageValues > 0'})).Revenue.agg(sessions='size',buyers='sum',buyer_rate='mean')
display(pv)
print('The above difference is descriptive. PageValues is excluded from operational predictors because its availability before a hypothetical decision is uncertain, not because association proves leakage.')

## 6. Four operational baselines

**Purpose.** Evaluate the taught families using the 16 decision-time candidates.

**Method.** Compare baseline ROC-AUC/AP on identical folds; quantify the AUC change from excluding `PageValues`.

**Implication.** Baselines provide the reference for targeted modifications.


In [ ]:
# A6 — four operational 16-source-predictor baselines
for family in ['Logit','GaussianNB','BernoulliNB','LDA']:
    label=f'Operational16 | {family}'
    print('Fitting',label,flush=True)
    RESULTS[label]=run_spec(label,family,SAFE)
    FOLD_ROWS.extend(RESULTS[label]['folds']); WARNING_ROWS.extend(RESULTS[label]['warnings'])
OP16=pd.DataFrame([{'candidate':r['candidate'],'scoring status':r['status'],'numerical status':compact_numerical_label(r),'mean ROC-AUC':r.get('mean_fold_auc',np.nan),
 'AUC SD':r.get('fold_auc_sd',np.nan),'AUC SE':r.get('fold_auc_se',np.nan),'mean AP':r.get('mean_fold_ap',np.nan),
 'pooled OOF AUC':r.get('pooled_oof_auc',np.nan),'warnings':len(r['warnings']),'failure':r['failure']} for r in RESULTS.values() if r['stage']=='operational16']).set_index('candidate')
display(OP16[['mean ROC-AUC','AUC SD','mean AP']].round(4))
op16_fold_diag=pd.DataFrame([x for r in RESULTS.values() if r['stage']=='operational16' for x in r['folds']])
op16_audit=op16_fold_diag.groupby('candidate').agg(rank_min=('design_rank','min'),rank_max=('design_rank','max'),
    encoded_columns_min=('design_columns','min'),encoded_columns_max=('design_columns','max'),
    all_parameters_finite=('finite_model_parameters','all'),all_probabilities_in_range=('probabilities_in_0_1','all'))
# Direct same-fold AUC changes quantify the effect of omitting PageValues in each family.
comparison=[]
for family in ['Logit','GaussianNB','BernoulliNB','LDA']:
    a=RESULTS[f'All17 | {family}']; b=RESULTS[f'Operational16 | {family}']
    if a['status']=='SCORED' and b['status']=='SCORED':
        for x,y in zip(a['folds'],b['folds']): comparison.append({'family':family,'fold':x['fold'],'AUC all17':x['roc_auc'],'AUC op16':y['roc_auc'],'op16 minus all17':y['roc_auc']-x['roc_auc']})
PAGE_COMPARE=pd.DataFrame(comparison)
PAGE_SUMMARY=PAGE_COMPARE.groupby('family')['op16 minus all17'].agg(['mean','std']).rename(columns={'mean':'mean AUC delta','std':'fold SD'}).round(4)
display(PAGE_SUMMARY[['mean AUC delta']])


**Interpretation.** Numerical checks are retained internally. Rare nominal levels can make unpenalised Logit coefficients non-identifiable, even if probabilities can be computed.


## 7. EDA-informed revisions — what changed and why

**Purpose.** Compare exactly three motivated revisions against the four original baselines. The aim is **predictive ROC-AUC**, not coefficient inference, and adding regularisation is a hypothesis rather than an automatic improvement.

| Revised model | Changes from operational baseline | Training EDA motivation | Hypothesis and limitation |
|---|---|---|---|
| **Logit** | Log1p six activity magnitudes; fit-fold >0 indicators for ≥50%-zero activity columns; fold-fitted StandardScaler; L2 penalty C=1.0 (fixed in advance) | Heavy activity tails/zeros; high within-class BounceRates–ExitRates and product-count/duration correlations; sparse pure outcome category levels | Lower coefficient instability/separation risk and potentially better unseen ranking. Added shrinkage can also introduce harmful bias; the revision's changes cannot be attributed individually. |
| **LDA** | Log1p six activity magnitudes; shared covariance **shrinkage='auto'**, solver='lsqr' | Heavy tails and strongly correlated numeric activity measures; mixed continuous/indicator design | More stable covariance estimates might improve prediction. Correlation is **allowed** by LDA and alone does not violate its assumptions; shrinkage may worsen AUC. |
| **Mixed Naive Bayes** | Gaussian log-activity and rate likelihoods + categorical nominal/Weekend/SpecialDay-presence likelihoods; Laplace α=1 for discrete categories | Right-skewed numeric activity, many exact zeros and genuinely nominal predictors | Matches data types more naturally than all-Gaussian or all-Bernoulli NB; conditional independence remains questionable. No additional regularisation experiment for NB. |

**Predeclared settings.** L2 strength C=1.0 and LDA automatic covariance shrinkage are fixed *before* comparing candidates. All seven use the same five training folds; all fitted scalers, zero-indicators, encoders and covariance estimates stay within each fold. No post-hoc regularisation search and no test-set selection.

**Interpretation.** Compare each revised specification with its original baseline below. An observed difference is for the **whole revision recipe**, not causal evidence that regularisation by itself helped.


In [ ]:
REVISION_SPECS=[('Revised | Logit','Logit','Logit'),
                 ('Revised | LDA','LDA','LDA'),
                 ('Revised | MixedNB','MixedNB','MixedNB')]
for label,family,kind in REVISION_SPECS:
    print('Fitting',label,flush=True)
    RESULTS[label]=run_spec(label,family,SAFE,pipeline_factory=lambda kind=kind:make_revision_pipeline(kind))
    FOLD_ROWS.extend(RESULTS[label]['folds'])
    WARNING_ROWS.extend(RESULTS[label]['warnings'])

for label,_,_ in REVISION_SPECS:
    result=RESULTS[label]
    if result.get('status')!='SCORED' or len(result.get('folds',[]))!=K or not np.isfinite(np.asarray(result.get('oof'),dtype=float)).all():
        raise RuntimeError(f'{label}: scoring failed or OOF coverage is incomplete; stop before comparisons.')
    if not np.all((np.asarray(result['oof'],dtype=float)>=0)&(np.asarray(result['oof'],dtype=float)<=1)):
        raise RuntimeError(f'{label}: OOF probabilities are outside [0,1].')

revision_comparisons=[]
for revised,baseline in [('Revised | Logit','Operational16 | Logit'),
                         ('Revised | LDA','Operational16 | LDA'),
                         ('Revised | MixedNB','Operational16 | GaussianNB'),
                         ('Revised | MixedNB','Operational16 | BernoulliNB')]:
    rr=RESULTS[revised]; bb=RESULTS[baseline]
    revision_comparisons.append({'revision':revised,'baseline':baseline,
      'revision_mean_auc':rr['mean_fold_auc'],'baseline_mean_auc':bb['mean_fold_auc'],
      'mean_auc_delta':rr['mean_fold_auc']-bb['mean_fold_auc'],
      'revision_mean_ap':rr['mean_fold_ap'],'baseline_mean_ap':bb['mean_fold_ap'],
      'mean_ap_delta':rr['mean_fold_ap']-bb['mean_fold_ap'],
      'revision_auc_fold_sd':rr['fold_auc_sd'],'baseline_auc_fold_sd':bb['fold_auc_sd']})
REVISION_COMPARISON=pd.DataFrame(revision_comparisons)
display(REVISION_COMPARISON[['revision','baseline','mean_auc_delta','mean_ap_delta']].round(4))
REVISION_COMPARISON.to_csv(ARTIFACTS/'revision_baseline_comparisons.csv',index=False)
revised_logit_folds=pd.DataFrame(RESULTS['Revised | Logit']['folds'])
revised_logit_folds.to_csv(ARTIFACTS/'revised_logit_numerical_gate.csv',index=False)
mixed_nb_audit=[{'candidate':'Revised | MixedNB','fold':int(r['fold']),**r['mixed_nb_component_audit']} for r in RESULTS['Revised | MixedNB']['folds']]
with (ARTIFACTS/'mixed_nb_component_audit.json').open('w') as f: json.dump(mixed_nb_audit,f,indent=2,allow_nan=False); f.write('\n')
mixed_audit_example_count=sum(len(r['validation_component_examples']) for r in mixed_nb_audit)
if not revised_logit_folds.numerical_status.eq('L2_REGULARISED_FINITE_FIT').all():
    raise RuntimeError('Revised L2 Logit had an invalid numerical status; stop before selection.')
if any(w['warning_type']=='ConvergenceWarning' for w in RESULTS['Revised | Logit']['warnings']):
    raise RuntimeError('Revised L2 Logit emitted a convergence warning.')


## 8. Model comparison and selection

**Purpose.** Choose the strongest operational classifier.

**Method.** Show CV AUC, fold SD and mean AP for seven candidates; use mean AUC for selection.

**Implication.** Model ranking does not choose a voucher threshold, and small differences are not formal superiority tests.


In [ ]:
OPERATIONAL_LABELS=['Operational16 | Logit','Operational16 | GaussianNB',
    'Operational16 | BernoulliNB','Operational16 | LDA',
    'Revised | Logit','Revised | LDA','Revised | MixedNB']
# Admission gate: seven complete operational OOF vectors; IRLS gate applies only to the two unpenalised Logit counterparts.
if len(OPERATIONAL_LABELS)!=7 or len(set(OPERATIONAL_LABELS))!=7:
    raise RuntimeError('The operational comparison must contain exactly seven distinct candidates.')
for label in OPERATIONAL_LABELS:
    r=RESULTS[label]; oof=np.asarray(r.get('oof'),dtype=float)
    if r.get('status')!='SCORED' or len(r.get('folds',[]))!=K or oof.shape!=(len(ytr),):
        raise RuntimeError(f'{label}: failed scoring or missing OOF/fold evidence.')
    if not np.isfinite(oof).all() or not np.all((oof>=0)&(oof<=1)):
        raise RuntimeError(f'{label}: OOF probabilities are non-finite or outside [0,1].')
    coverage=np.zeros(len(ytr),dtype=int)
    for expected_fold,((fit_idx,val_idx),fold) in enumerate(zip(FOLD_INDICES,r['folds']),1):
        if int(fold.get('fold',-1))!=expected_fold or fold.get('fit_status')!='OK' or int(fold.get('n_validation',-1))!=len(val_idx):
            raise RuntimeError(f'{label}: invalid fold status or validation coverage in fold {expected_fold}.')
        if not fold.get('finite_model_parameters',False) or not fold.get('finite_probabilities',False) or not fold.get('probabilities_in_0_1',False):
            raise RuntimeError(f'{label}: non-finite fitted parameters or probabilities in fold {expected_fold}.')
        coverage[val_idx]+=1
        yv=ytr.iloc[val_idx].to_numpy(); pv=oof[val_idx]
        if (not np.isclose(roc_auc_score(yv,pv),float(fold['roc_auc']),rtol=0,atol=1e-12)
            or not np.isclose(average_precision_score(yv,pv),float(fold['average_precision']),rtol=0,atol=1e-12)):
            raise RuntimeError(f'{label}: saved fold scores do not match its OOF vector in fold {expected_fold}.')
    if not np.all(coverage==1):
        raise RuntimeError(f'{label}: every training row must have exactly one validation prediction.')
for label in ['All17 | Logit','Operational16 | Logit']:
    for fold in RESULTS[label]['folds']:
        if (not bool(fold.get('reference_passes_fixed_criteria',False))
            or not bool(fold.get('finite_model_parameters',False))
            or not bool(fold.get('finite_probabilities',False))
            or not bool(fold.get('probabilities_in_0_1',False))):
            raise RuntimeError(f'{label}: required finite-parameter/probability or IRLS gate failed in fold {fold.get("fold")}; stop before winner selection.')

cv_rows=[]
for label in OPERATIONAL_LABELS:
    r=RESULTS[label]
    cv_rows.append({'candidate':label,'mean_fold_auc':r['mean_fold_auc'],
      'auc_fold_sd':r['fold_auc_sd'],'auc_fold_se':r['fold_auc_se'],
      'mean_fold_ap':r['mean_fold_ap'],'ap_fold_sd':r['fold_ap_sd'],'ap_fold_se':r['fold_ap_se'],
      'numerical_status':compact_numerical_label(r),'warning_count':len(r['warnings'])})
SEVEN_CV=pd.DataFrame(cv_rows)
max_auc=float(SEVEN_CV.mean_fold_auc.max())
exact_ties=sorted(SEVEN_CV.loc[SEVEN_CV.mean_fold_auc.eq(max_auc),'candidate'].tolist())
WINNER_LABEL=exact_ties[0]
WINNER_SE=float(SEVEN_CV.loc[SEVEN_CV.candidate.eq(WINNER_LABEL),'auc_fold_se'].iloc[0])
SEVEN_CV['within_winner_one_se']=SEVEN_CV.mean_fold_auc.ge(max_auc-WINNER_SE)
SEVEN_CV['selected_by_primary_auc']=SEVEN_CV.candidate.eq(WINNER_LABEL)
display(SEVEN_CV[['candidate','mean_fold_auc','auc_fold_sd','mean_fold_ap']].round(4))
SEVEN_CV.to_csv(ARTIFACTS/'seven_candidate_cv.csv',index=False)

fold_score_rows=[]; oof_rows=[]
for label in OPERATIONAL_LABELS:
    r=RESULTS[label]
    for fold in r['folds']:
        fold_score_rows.append({'candidate':label,'fold':int(fold['fold']),
          'roc_auc':float(fold['roc_auc']),'average_precision':float(fold['average_precision']),
          'n_validation':int(fold['n_validation'])})
    for row_id,y_true,score in zip(Xtr.index.to_numpy(),ytr.to_numpy(),r['oof']):
        oof_rows.append({'candidate':label,'row_id':int(row_id),'observed':int(y_true),
          'oof_probability':float(score)})
SEVEN_FOLD_SCORES=pd.DataFrame(fold_score_rows)
SEVEN_OOF=pd.DataFrame(oof_rows)
SEVEN_FOLD_SCORES.to_csv(ARTIFACTS/'seven_candidate_fold_scores.csv',index=False)
SEVEN_OOF.to_csv(ARTIFACTS/'seven_candidate_oof_predictions.csv',index=False,float_format='%.17g')

fig,ax=plt.subplots(figsize=(7.2,6))
for label in OPERATIONAL_LABELS:
    fpr,tpr,_=roc_curve(ytr,RESULTS[label]['oof'],drop_intermediate=False)
    ax.plot(fpr,tpr,lw=1.35,label=label)
ax.plot([0,1],[0,1],color='black',lw=1,linestyle='--',label='Chance')
ax.set(xlabel='False positive rate',ylabel='True positive rate',
       title='Training OOF ROC curves — seven operational candidates')
ax.legend(fontsize=7,loc='lower right'); ax.grid(alpha=.2); fig.tight_layout()
fig.savefig(ARTIFACTS/'training_oof_roc_seven.png',dpi=180,bbox_inches='tight')
display(fig); plt.close(fig)
WINNER_OOF=np.asarray(RESULTS[WINNER_LABEL]['oof'],dtype=float)
print(f'Winner by mean five-fold ROC-AUC: {WINNER_LABEL} (AUC {max_auc:.6f}, fold SE {WINNER_SE:.6f}).')
for _,r in SEVEN_CV.iterrows():
    print(f"{r.candidate}: AUC gap from winner {max_auc-r.mean_fold_auc:.6f}; within selected winner one-SE range={bool(r.within_winner_one_se)}.")


## 9. OOF predicted probabilities

**Purpose.** Inspect the selected model's score distribution.

**Method.** Plot training OOF buyer probabilities by observed class.

**Implication.** Low score concentration explains why high cutoffs issue vouchers almost universally.


In [ ]:
fig,ax=plt.subplots(figsize=(7.2,4.5))
for cls,color,name in [(0,'#4477AA','Observed non-buyer'),(1,'#CC6677','Observed buyer')]:
    ax.hist(WINNER_OOF[np.asarray(ytr)==cls],bins=np.linspace(0,1,31),
            density=True,alpha=.48,color=color,label=name)
ax.set(xlabel='Training OOF predicted purchase probability',ylabel='Density',
       title=f'OOF probability distribution — {WINNER_LABEL}')
ax.legend(); ax.grid(axis='y',alpha=.2); fig.tight_layout()
fig.savefig(ARTIFACTS/'training_winner_oof_probability_histogram.png',dpi=180,bbox_inches='tight')
display(fig); plt.close(fig)
hist_rows=[]
for cls,name in [(0,'Observed non-buyer'),(1,'Observed buyer')]:
    values=WINNER_OOF[np.asarray(ytr)==cls]
    hist_rows.append({'observed_group':name,'sessions':len(values),'mean_probability':float(values.mean()),
      'median_probability':float(np.median(values)),'share_below_0.10':float(np.mean(values<.10)),
      'share_below_0.50':float(np.mean(values<.50)),'share_above_0.90':float(np.mean(values>.90)),
      'maximum_probability':float(values.max())})
WINNER_HIST_SUMMARY=pd.DataFrame(hist_rows)
hist_nonbuyer=WINNER_HIST_SUMMARY.loc[WINNER_HIST_SUMMARY.observed_group.eq('Observed non-buyer')].iloc[0]
hist_buyer=WINNER_HIST_SUMMARY.loc[WINNER_HIST_SUMMARY.observed_group.eq('Observed buyer')].iloc[0]
share_high=max(hist_nonbuyer['share_above_0.90'],hist_buyer['share_above_0.90'])
print(f"Nonbuyer OOF mean/median={hist_nonbuyer.mean_probability:.4f}/{hist_nonbuyer.median_probability:.4f}; buyer mean/median={hist_buyer.mean_probability:.4f}/{hist_buyer.median_probability:.4f}. Scores are below .50 for {hist_nonbuyer['share_below_0.50']:.1%} of nonbuyers and {hist_buyer['share_below_0.50']:.1%} of buyers; at most {share_high:.1%} exceed .90 and the largest is {max(hist_nonbuyer['maximum_probability'],hist_buyer['maximum_probability']):.3f}. This concentration explains very few buyer predictions and near-universal voucher coverage at the fixed 2/3 cutoff, but does not establish calibration.")

## 10. Four threshold references

**Purpose.** Compare buyer detection, specificity and voucher coverage.

**Method.** Use training prevalence, 0.50, the precommitted 2/3 and the sensitivity–specificity crossover; voucher if \(\hat p<t\).

**Scenario.** A 20% voucher and assumed \(q=0.50\) imply the cost-based reference \(t=0.40/(0.40+0.20)=2/3\) if probabilities are meaningful.

**Implication.** The crossover balances sensitivity and specificity; neither cutoff is a verified commercial optimum.


In [ ]:
POLICY_DISCOUNT=.20; Q_REFERENCE=.50
C_FN_PER_A=POLICY_DISCOUNT
C_FP_PER_A=(1-POLICY_DISCOUNT)*Q_REFERENCE
COST_THRESHOLD=C_FP_PER_A/(C_FP_PER_A+C_FN_PER_A)
if not np.isclose(COST_THRESHOLD,2/3,rtol=0,atol=1e-15):
    raise AssertionError('The fixed cost derivation must equal 2/3.')
print(f'With d={POLICY_DISCOUNT:.2f} and assumed q_ref={Q_REFERENCE:.2f}: C_FN/A={C_FN_PER_A:.2f}, C_FP/A={C_FP_PER_A:.2f}, t*=C_FP/(C_FP+C_FN)={COST_THRESHOLD:.8f}; this rationale requires meaningful probabilities.')

def threshold_metrics(y_true,probability,threshold,q=.50,discount=.20):
    y=np.asarray(y_true,dtype=int); p=np.asarray(probability,dtype=float)
    pred=(p>=threshold).astype(int); voucher=(pred==0)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    precision=float(tp/(tp+fp)) if (tp+fp)>0 else None
    sensitivity=float(tp/(tp+fn)) if (tp+fn)>0 else np.nan
    specificity=float(tn/(tn+fp)) if (tn+fp)>0 else np.nan
    r0=float(tp+fn)
    rq=float(tp+(1-discount)*fn+(1-discount)*q*tn)
    delta=rq-r0
    direct=np.where(y==1,np.where(voucher,1-discount,1.0),
                    np.where(voucher,(1-discount)*q,0.0)).sum()
    if not np.isclose(rq,direct,rtol=0,atol=1e-10):
        raise AssertionError('Voucher formula disagrees with row-level expected revenue.')
    return {'threshold':float(threshold),'sensitivity':sensitivity,'specificity':specificity,
      'precision':precision,'TP':int(tp),'FP':int(fp),'FN':int(fn),'TN':int(tn),
      'voucher_count':int(voucher.sum()),'voucher_share':float(voucher.mean()),
      'R0_over_A':r0,'Rq_over_A':rq,'delta_R_over_A':delta,
      'relative_delta_vs_R0':(delta/r0 if r0 else None),
      'direct_expected_revenue_over_A':float(direct)}

roc_fpr,roc_tpr,roc_thresholds=roc_curve(ytr,WINNER_OOF,drop_intermediate=False)
crossover_rows=[]
for fpr,tpr,t in zip(roc_fpr,roc_tpr,roc_thresholds):
    if np.isfinite(t) and 0<=t<=1:
        crossover_rows.append({'threshold':float(t),'sensitivity':float(tpr),
          'specificity':float(1-fpr),'absolute_rate_gap':float(abs(tpr-(1-fpr)))})
# Include fixed probability endpoints; calculate rates only, without policy/revenue arithmetic.
crossover_rows.append({'threshold':0.0,'sensitivity':1.0,'specificity':0.0,'absolute_rate_gap':1.0})
pred_at_one=(WINNER_OOF>=1.0).astype(int)
tn1,fp1,fn1,tp1=confusion_matrix(ytr,pred_at_one,labels=[0,1]).ravel()
sens1=float(tp1/(tp1+fn1)); spec1=float(tn1/(tn1+fp1))
crossover_rows.append({'threshold':1.0,'sensitivity':sens1,'specificity':spec1,
  'absolute_rate_gap':abs(sens1-spec1)})
CROSSOVER_ROW=min(crossover_rows,key=lambda r:(r['absolute_rate_gap'],r['threshold']))
CROSSOVER_THRESHOLD=float(CROSSOVER_ROW['threshold'])
THRESHOLD_REFERENCES=[
    ('Training purchase base rate',float(np.mean(ytr))),
    ('Conventional 0.500',.5),
    ('Primary cost-based 2/3',2/3),
    ('Sensitivity-specificity crossover',CROSSOVER_THRESHOLD)]
THRESHOLD_TABLE=pd.DataFrame([
    {'reference':label,**threshold_metrics(ytr,WINNER_OOF,t,q=Q_REFERENCE,discount=POLICY_DISCOUNT)}
    for label,t in THRESHOLD_REFERENCES])
THRESHOLD_TABLE.to_csv(ARTIFACTS/'winner_four_thresholds.csv',index=False)
THRESHOLD_CURVE=pd.DataFrame(crossover_rows).sort_values('threshold').reset_index(drop=True)
THRESHOLD_CURVE.to_csv(ARTIFACTS/'winner_threshold_sensitivity_curve.csv',index=False)
display(THRESHOLD_TABLE[['reference','threshold','TP','sensitivity','specificity','voucher_share','delta_R_over_A']].round(4))
primary_threshold_row=THRESHOLD_TABLE.loc[THRESHOLD_TABLE.reference.eq('Primary cost-based 2/3')].iloc[0]
fig,ax=plt.subplots(figsize=(7.2,4.8))
ax.plot(THRESHOLD_CURVE.threshold,THRESHOLD_CURVE.sensitivity,label='Sensitivity',lw=1.6)
ax.plot(THRESHOLD_CURVE.threshold,THRESHOLD_CURVE.specificity,label='Specificity',lw=1.6)
reference_styles=[('#999999','--',1.0),('#4477AA',':',1.0),('#CC3311','-',2.2),('#228833','-.',1.4)]
for (label,t),(color,style,width) in zip(THRESHOLD_REFERENCES,reference_styles):
    ax.axvline(t,color=color,lw=width,alpha=.8,linestyle=style,label=f'{label} (t={t:.4f})')
ax.scatter([CROSSOVER_THRESHOLD],[CROSSOVER_ROW['sensitivity']],s=38,marker='o',
           color='#228833',zorder=5,label='Crossover point')
ax.set(xlabel='Buyer threshold (voucher if p < threshold)',ylabel='Rate',ylim=(0,1.02),
       title=f'Training OOF threshold sensitivity — {WINNER_LABEL}')
ax.grid(alpha=.2); ax.legend(fontsize=7,ncol=2); fig.tight_layout()
fig.savefig(ARTIFACTS/'winner_threshold_sensitivity.png',dpi=180,bbox_inches='tight')
display(fig); plt.close(fig)


## 11. Hypothetical conversion scenarios

**Purpose.** Show dependence of voucher arithmetic on an unobserved assumption.

**Method.** Hold the 2/3 assignments fixed and vary incremental conversion \(q\) across 10%, 25%, 50% and 100%.

**Implication.** Simulated revenue is not measured sales lift or profit.


In [ ]:
PRIMARY_THRESHOLD=2/3; DISCOUNT=.20; Q_REFERENCE=.50; Q_SCENARIOS=[.10,.25,.50,1.00]
PRIMARY_POLICY=threshold_metrics(ytr,WINNER_OOF,PRIMARY_THRESHOLD,q=Q_REFERENCE,discount=DISCOUNT)
primary_pred=(WINNER_OOF>=PRIMARY_THRESHOLD).astype(int); primary_voucher=(primary_pred==0)
tn,fp,fn,tp=(PRIMARY_POLICY[k] for k in ['TN','FP','FN','TP'])
q_rows=[]
for q in Q_SCENARIOS:
    r0=float(tp+fn); rq=float(tp+(1-DISCOUNT)*fn+(1-DISCOUNT)*q*tn); delta=rq-r0
    direct=np.where(np.asarray(ytr)==1,np.where(primary_voucher,1-DISCOUNT,1.0),
                    np.where(primary_voucher,(1-DISCOUNT)*q,0.0)).sum()
    if not np.isclose(rq,direct,rtol=0,atol=1e-10):
        raise AssertionError(f'q={q}: formula/direct arithmetic mismatch')
    q_rows.append({'q_incremental_conversion':q,'threshold':PRIMARY_THRESHOLD,'discount':DISCOUNT,
      'voucher_count_fixed':PRIMARY_POLICY['voucher_count'],'voucher_share_fixed':PRIMARY_POLICY['voucher_share'],
      'R0_over_A':r0,'Rq_over_A':rq,'delta_R_over_A':delta,
      'relative_delta_vs_R0':(delta/r0 if r0 else None),'direct_expected_revenue_over_A':float(direct)})
Q_SCENARIO_TABLE=pd.DataFrame(q_rows)
Q_SCENARIO_TABLE.to_csv(ARTIFACTS/'winner_q_scenarios_at_two_thirds.csv',index=False)
display(Q_SCENARIO_TABLE[['q_incremental_conversion','voucher_count_fixed','Rq_over_A','delta_R_over_A']].round(2))
fig,ax=plt.subplots(figsize=(6.6,4.2))
ax.plot(Q_SCENARIO_TABLE.q_incremental_conversion,Q_SCENARIO_TABLE.Rq_over_A,
        marker='o',label='Expected voucher revenue')
ax.axhline(Q_SCENARIO_TABLE.R0_over_A.iloc[0],color='black',linestyle='--',
           label='No-voucher reference')
ax.set(xlabel='Assumed incremental conversion probability q',ylabel='Hypothetical expected revenue / A',
       title='Training OOF: fixed 2/3 hypothetical revenue sensitivity')
ax.grid(alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig(ARTIFACTS/'winner_q_scenario_sensitivity.png',dpi=180,bbox_inches='tight')
display(fig); plt.close(fig)
print(f'At 2/3: TP={tp}, FP={fp}, FN={fn}, TN={tn}; fixed voucher count={PRIMARY_POLICY["voucher_count"]}.')

def json_clean(v):
    if isinstance(v,dict): return {str(k):json_clean(x) for k,x in v.items()}
    if isinstance(v,(list,tuple)): return [json_clean(x) for x in v]
    if isinstance(v,(np.integer,)): return int(v)
    if isinstance(v,(np.floating,float)):
        x=float(v); return x if np.isfinite(x) else None
    if isinstance(v,(np.bool_,)): return bool(v)
    return v

zero_shares={c:float(Xtr[c].eq(0).mean()) for c in ACTIVITY}
recipe_details={
 'Operational16 | Logit':{'family':'Logit','C':'Infinity','solver':'newton-cg','tol':1e-8,'max_iter':2000,'representation':'Raw numeric magnitudes including SpecialDay; six nominal fields k-1 OHE; Weekend 0/1.'},
 'Operational16 | GaussianNB':{'family':'GaussianNB','var_smoothing':'scikit-learn default','representation':'Raw numeric magnitudes including SpecialDay; six nominal fields full k OHE; Weekend 0/1.'},
 'Operational16 | BernoulliNB':{'family':'BernoulliNB','alpha':1.0,'binarize':None,'representation':'Numeric >0 indicators; six nominal fields full k OHE; Weekend 0/1.'},
 'Operational16 | LDA':{'family':'LDA','solver':'svd','representation':'Raw numeric magnitudes including SpecialDay; six nominal fields k-1 OHE; Weekend 0/1.'},
 'Revised | Logit':{'family':'Logit','C':1.0,'penalty':'L2','solver':'lbfgs','tol':1e-8,'max_iter':3000,'standardize':'StandardScaler fitted on training fold only (all encoded features)','representation':'All 16 sources; log1p six activity columns; add >0 indicators only when fit-fold zero share >=.50; raw BounceRates/ExitRates/SpecialDay/Weekend; k-1 OHE; scale encoded columns for an equal-unit L2 penalty.',
   'fold_zero_shares':{str(r['fold']):json.loads(r['fold_zero_share']) for r in RESULTS['Revised | Logit']['folds']},
   'fold_presence_columns':{str(r['fold']):json.loads(r['fold_presence_columns']) for r in RESULTS['Revised | Logit']['folds']}},
 'Revised | LDA':{'family':'LDA','solver':'lsqr','shrinkage':'auto (Ledoit-Wolf, estimated on fit fold)','representation':'All 16 sources; log1p six activity columns; raw BounceRates/ExitRates/SpecialDay/Weekend; k-1 OHE; shrink class covariance toward diagonal for stability.'},
 'Revised | MixedNB':{'family':'MixedGaussianCategoricalNB','alpha':1.0,'GaussianNB_var_smoothing':'scikit-learn default',
   'prior':'Empirical prior appears once in Gaussian posterior; categorical likelihoods are added and normalized by logsumexp.',
   'representation':'All 16 sources; Gaussian block log1p six activity plus raw BounceRates/ExitRates; categorical block six nominal + Weekend 0/1 + SpecialDay>0, fit-fold maps reserve unknown code 0 and Laplace support.'}}
train_ids=Xtr.index.to_numpy(dtype=int).tolist()
reserved_test_ids=[int(x) for x in test_ids]
fold_ids=[{'fold':i+1,'fit_row_ids':Xtr.index.to_numpy()[fit].astype(int).tolist(),
           'validation_row_ids':Xtr.index.to_numpy()[val].astype(int).tolist()}
          for i,(fit,val) in enumerate(FOLD_INDICES)]
freeze={
 'status':'REGULARISATION REVISION: TRAINING-CV COMPARISON ONLY; NO FRESH TEST AUDIT',
 'pretest_review':'Prior 2026-10-25 PASS applied only to the earlier, unregularised model recipes',
 'winner_label':WINNER_LABEL,'winner_exact_recipe':recipe_details[WINNER_LABEL],
 'operational_selection_rule':'Highest mean five-fold ROC-AUC across exactly seven operational candidates; exact ties broken alphabetically by candidate label. One-SE proximity is descriptive only; AP and voucher revenue do not select.',
 'seven_candidate_cv':SEVEN_CV.to_dict(orient='records'),
 'seed':SEED,'test_size':TEST_FRAC,'training_row_ids':train_ids,'reserved_test_row_ids':reserved_test_ids,
 'fold_ids':fold_ids,'source_csv_sha256':source_hash,'source_columns':SAFE,
 'primary_threshold':PRIMARY_THRESHOLD,'discount':DISCOUNT,'q_reference':Q_REFERENCE,
 'threshold_reference_values':{label:float(t) for label,t in THRESHOLD_REFERENCES},
 'primary_policy_training_oof_counts':{k:int(PRIMARY_POLICY[k]) for k in ['TP','FP','FN','TN','voucher_count']},
 'training_policy_and_scenarios_use_oof_only':True,'test_predictions_or_outcomes_accessed':False}
freeze=json_clean(freeze)
with (ARTIFACTS/'training_freeze.json').open('w') as f:
    json.dump(freeze,f,indent=2,allow_nan=False); f.write('\n')


In [ ]:
# Persist the reviewed training-stage candidate evidence before the single-winner holdout audit.
summary_rows=[]
for label,r in RESULTS.items():
    summary_rows.append({'candidate':label,'stage':r['stage'],'scoring_status':r['status'],'numerical_status':r['numerical_status'],
      'mean_fold_auc':r.get('mean_fold_auc',np.nan),'fold_auc_sd':r.get('fold_auc_sd',np.nan),'fold_auc_se':r.get('fold_auc_se',np.nan),
      'mean_fold_ap':r.get('mean_fold_ap',np.nan),'fold_ap_sd':r.get('fold_ap_sd',np.nan),'fold_ap_se':r.get('fold_ap_se',np.nan),
      'pooled_oof_auc':r.get('pooled_oof_auc',np.nan),'pooled_oof_ap':r.get('pooled_oof_ap',np.nan),'warning_count':len(r['warnings']),'failure':r['failure']})
model_summary=pd.DataFrame(summary_rows)
baseline_summary=model_summary[~model_summary.candidate.str.startswith('Revised | ')].copy()
baseline_folds=pd.DataFrame(FOLD_ROWS)
baseline_folds=baseline_folds[~baseline_folds.candidate.str.startswith('Revised | ')].copy()
model_warnings=pd.DataFrame(WARNING_ROWS,columns=['candidate','fold','warning_type','message'])
baseline_warnings=model_warnings[~model_warnings.candidate.str.startswith('Revised | ')].copy()
baseline_summary.to_csv(ARTIFACTS/'baseline_summary.csv',index=False)
model_summary.to_csv(ARTIFACTS/'model_summary.csv',index=False)
with (ARTIFACTS/'baseline_summary.json').open('w') as f: json.dump(json_clean(baseline_summary.to_dict(orient='records')),f,indent=2,allow_nan=False); f.write('\n')
with (ARTIFACTS/'model_summary.json').open('w') as f: json.dump(json_clean(model_summary.to_dict(orient='records')),f,indent=2,allow_nan=False); f.write('\n')
baseline_folds.to_csv(ARTIFACTS/'baseline_fold_metrics_diagnostics.csv',index=False)
model_summary_folds=pd.DataFrame(FOLD_ROWS)
model_summary_folds.to_csv(ARTIFACTS/'model_fold_metrics_diagnostics.csv',index=False)
baseline_warnings.to_csv(ARTIFACTS/'baseline_warnings.csv',index=False)
model_warnings.to_csv(ARTIFACTS/'model_warnings.csv',index=False)
for label,r in RESULTS.items():
    if r['oof'] is not None:
        slug_map={'All17 | Logit':'all17_Logit','All17 | GaussianNB':'all17_gaussian_nb','All17 | BernoulliNB':'all17_bernoulli_nb','All17 | LDA':'all17_LDA',
          'Operational16 | Logit':'operational16_Logit','Operational16 | GaussianNB':'operational16_gaussian_nb','Operational16 | BernoulliNB':'operational16_bernoulli_nb','Operational16 | LDA':'operational16_LDA',
          'Revised | Logit':'revised_Logit','Revised | LDA':'revised_LDA','Revised | MixedNB':'revised_MixedNB'}
        slug=slug_map[label]
        pd.DataFrame({'row_id':Xtr.index,'observed':ytr.to_numpy(),'oof_probability':r['oof']}).to_csv(ARTIFACTS/(slug+'_oof.csv'),index=False,float_format='%.17g')


## 12. Training-only selected-model stability check

**Purpose.** Confirm that the CV-selected candidate can be refitted on all training rows without numerical failure.

**Method.** Use exactly the selected specification and fit only the 9,864 training sessions. Check finite parameters and probabilities and record model-specific estimator settings. The historical 20% holdout is **not** scored again.

**Interpretation.** This is a training-fit audit, not an unbiased estimate of generalisation. Previous test AUC and voucher counts refer to a different, unregularised model specification.


In [ ]:
# Current candidate comparison is training-only because the historical holdout was previously viewed.
FREEZE=json.loads((ARTIFACTS/'training_freeze.json').read_text())
if FREEZE['winner_label']!=WINNER_LABEL or len(FREEZE['seven_candidate_cv'])!=7:
    raise RuntimeError('Training selection provenance differs from the current seven-candidate comparison.')

def instantiate_selected(label):
    if label=='Revised | Logit': return make_revision_pipeline('Logit')
    if label=='Revised | LDA': return make_revision_pipeline('LDA')
    if label=='Revised | MixedNB': return make_revision_pipeline('MixedNB')
    prefixes={'Operational16 | Logit':'Logit','Operational16 | GaussianNB':'GaussianNB',
              'Operational16 | BernoulliNB':'BernoulliNB','Operational16 | LDA':'LDA'}
    if label not in prefixes: raise ValueError('Unknown operational model: '+label)
    return make_pipeline(SAFE,prefixes[label])

WINNER_PIPELINE=instantiate_selected(WINNER_LABEL)
with warnings.catch_warnings(record=True) as audit_caught:
    warnings.simplefilter('always')
    WINNER_PIPELINE.fit(Xtr[SAFE],ytr)
WINNER_FIT_WARNINGS=[{'category':w.category.__name__,'message':str(w.message)[:300]} for w in audit_caught]
if any(issubclass(w.category,ConvergenceWarning) for w in audit_caught):
    raise RuntimeError('Selected candidate training refit did not converge.')
WINNER_EST=WINNER_PIPELINE.named_steps['clf']
WINNER_FINITE,WINNER_PARAMETER_CHECKS=finite_fitted_parameters(WINNER_EST)
WINNER_TRAIN_P=np.asarray(WINNER_PIPELINE.predict_proba(Xtr[SAFE])[:,np.flatnonzero(WINNER_EST.classes_==1)[0]],dtype=float)
if not WINNER_FINITE or not np.isfinite(WINNER_TRAIN_P).all() or np.any((WINNER_TRAIN_P<0)|(WINNER_TRAIN_P>1)):
    raise RuntimeError('Winner train refit produced invalid probabilities or parameters.')
TRAINING_WINNER_AUDIT={'selected_candidate':WINNER_LABEL,'training_rows':int(len(ytr)),
  'heldout_predictions_recomputed':False,'heldout_audit_status':'historical test viewed previously; not reused for revision selection',
  'estimator_class':type(WINNER_EST).__name__,'estimator_params':json_clean(WINNER_EST.get_params()),
  'finite_parameter_checks':WINNER_PARAMETER_CHECKS,'warnings':WINNER_FIT_WARNINGS,
  'in_sample_roc_auc_for_verification_only':float(roc_auc_score(ytr,WINNER_TRAIN_P))}
with (ARTIFACTS/'regularisation_training_winner_audit.json').open('w') as f:
    json.dump(json_clean(TRAINING_WINNER_AUDIT),f,indent=2,allow_nan=False); f.write('\n')
print(f'Training-only winner refit: {WINNER_LABEL}; {len(ytr):,} training rows; all parameters and probabilities finite. No held-out model scoring performed.')


**Interpretation.** The selected model is supported by common-fold training cross-validation, not an independent new holdout. The previous test figures in the 25 October archived verification report belong to the **previous unpenalised revision** and must not be used to claim improvement for the regularised revision.

**Caution.** Regularisation may increase or decrease ROC-AUC. Comparisons involving revised Logit and LDA jointly include their prior feature engineering; the observed deltas cannot isolate the contribution of the penalties.


## 13. Reproducibility and final-model boundary

**Purpose.** Record what is and is not final after this reopened candidate search.

**Method.** Save exact training-fit metadata and the selected recipe from the training comparison. Retain the previous held-out audit artifacts unchanged for historical reference.

**Implication.** The newly revised winner has **not** been given a new independent test audit. The old full-data model is not presented as an evaluation of the new candidate.


In [ ]:
# Save a transparent recipe and provenance note without reusing held-out outcomes.
FINAL_METADATA={
    'candidate':WINNER_LABEL,'selection':'maximum mean five-fold training CV ROC-AUC among exactly seven eligible candidates',
    'training_rows':int(len(ytr)),'historical_test_rows':int(len(test_ids)),
    'all_source_rows':12330,'no_new_heldout_scoring':True,
    'source_csv_sha256':source_hash,
    'recipe':freeze['winner_exact_recipe'] if (freeze:=FREEZE) else None,
    'train_refit_estimator':TRAINING_WINNER_AUDIT['estimator_class'],
    'candidate_CV_auc':float(SEVEN_CV.loc[SEVEN_CV.candidate.eq(WINNER_LABEL),'mean_fold_auc'].iloc[0]),
    'historical_test_results':'artifacts/test_metrics.json belongs to the prior unpenalised winning recipe, not this revision',
    'training_fit_finite':True
}
with (ARTIFACTS/'regularisation_revision_metadata.json').open('w') as f:
    json.dump(json_clean(FINAL_METADATA),f,indent=2,allow_nan=False); f.write('\n')
print('Recorded training-only winner provenance. No new test audit or all-data deployment refit.')


## 14. Conclusion and evidence boundaries

**Model selection.** Report the highest common-fold training CV ROC-AUC among the seven revised candidates, alongside AP and the original baseline comparison. No candidate was given a second-stage advantage.

**Revision attribution.** Logit's revision combines tail/zero handling, scaling and L2 regularisation; LDA's combines tail handling and covariance shrinkage; NB's combines appropriate data-type likelihoods and Laplace smoothing. Consequently, one cannot isolate the effect of regularisation from these comparisons.

**Limitations.** The 2/3 voucher exercise is conditional on hypothetical assumptions. Existing test results belong to the historical unpenalised model and cannot validate the updated candidates. Training-based candidate development may make CV results optimistic.


In [ ]:
# Training-only summary for the reopened, seven-candidate comparison.
WINNER_CV_ROW=SEVEN_CV.loc[SEVEN_CV.candidate.eq(WINNER_LABEL)].iloc[0]
MODEL_REVISIONS={
    'Revised | Logit':'log1p activity + fold-fitted presence + fold-only scaling + fixed L2 penalty C=1',
    'Revised | LDA':'log1p activity + automatic covariance shrinkage (lsqr solver)',
    'Revised | MixedNB':'mixed Gaussian/discrete likelihoods + Laplace categorical smoothing alpha=1'
}
PROJECT_SUMMARY={
    'project':'Online Shopper Purchase Prediction',
    'status':'revised seven-candidate training CV complete; no new independent holdout audited',
    'selected_candidate':WINNER_LABEL,
    'model_selection_criterion':'highest mean five-fold training ROC-AUC among seven fixed specifications',
    'mean_cv_auc':float(WINNER_CV_ROW.mean_fold_auc),
    'cv_auc_sd':float(WINNER_CV_ROW.auc_fold_sd),
    'cv_auc_se':float(WINNER_CV_ROW.auc_fold_se),
    'mean_cv_average_precision':float(WINNER_CV_ROW.mean_fold_ap),
    'all_seven_cv_auc':{row.candidate:float(row.mean_fold_auc) for row in SEVEN_CV.itertuples()},
    'revision_changes':MODEL_REVISIONS,
    'threshold_policy_training_only':float(PRIMARY_THRESHOLD),
    'training_oof_voucher_count':int(PRIMARY_POLICY['voucher_count']),
    'test_auc':None,'test_audit':'Previously evaluated historical test data; no new unbiased test score for these revisions.',
    'historical_test_metrics_not_comparable':True,
    'train_rows':int(len(ytr)),'heldout_rows_unscored_in_new_revision':int(len(test_ids))
}
PROJECT_SUMMARY=json_clean(PROJECT_SUMMARY)
with (ARTIFACTS/'regularisation_revision_summary.json').open('w') as f:
    json.dump(PROJECT_SUMMARY,f,indent=2,allow_nan=False); f.write('\n')
print('PROJECT SUMMARY — JSON')
print(json.dumps(PROJECT_SUMMARY,indent=2))
